# 04_pinecone_vector_database_all_videos

This notebook loads **all processed timestamped chunk files** from `../data/processed`, prepares rich metadata, creates embeddings, and upserts the vectors into **Pinecone**.

It is designed to be a clean continuation of your pipeline:

1. YouTube transcript collection  
2. Timestamp-aware chunking  
3. Corpus quality analysis  
4. **Vector database ingestion (this notebook)**  
5. Retrieval / chat / agent layer

## Why this version is better than the one-video prototype

This notebook is built for a full corpus, not a single video. It:

- discovers all `chunks_timestamped_*.json` files automatically
- preserves metadata needed later for **RAG**, **agents**, and **chat memory**
- stores **timestamps**, **video info**, and **source segment provenance**
- supports namespaces, batching, verification, and smoke-test querying
- writes an ingestion manifest you can reuse in later notebooks / apps

## Later compatibility with your final project

This notebook is intentionally shaped so it can plug into an end-to-end app later:

- **Vector database**: Pinecone namespace + metadata filters
- **Conversational chat interface**: chunk text + timestamps enable answer citations
- **Agentic setup**: metadata is rich enough for retriever tools and tool-routing
- **Memory**: the namespace / document ids are stable for retrieval workflows
- **Deployment**: the ingestion manifest can be reused in your web app backend

This notebook handles only the **vector store ingestion** stage.  
The later chat / agent notebook or app can sit on top of this index.

In [5]:
# Optional installs (run if needed)
# %pip install -q pinecone[grpc] openai python-dotenv pandas tqdm matplotlib

In [6]:
import os
import json
import glob
import hashlib
from datetime import datetime
from typing import List, Dict, Any

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

## Configuration

Adjust these values before running the notebook.

### Notes
- `PROCESSED_DIR` should point to your processed chunk JSON files.
- `CHUNK_GLOB` is set for the timestamped chunk files created earlier.
- `INDEX_NAME` should be lowercase and use only allowed Pinecone characters.
- `NAMESPACE` lets you isolate this corpus inside the same index.
- `EMBEDDING_MODEL` is set to `text-embedding-3-small` by default.

In [7]:
# ---------- Local paths ----------
PROCESSED_DIR = "../data/processed"
CHUNK_GLOB = "chunks_timestamped_*.json"
MANIFEST_FILENAME = "pinecone_ingestion_manifest.json"

# ---------- Pinecone ----------
INDEX_NAME = "youtube-rag-mechanical-engineering"
NAMESPACE = "efficient-engineer-v1"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# ---------- Embeddings ----------
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSIONS = 1536
OPENAI_EMBED_BATCH_SIZE = 100

# ---------- Upsert ----------
UPSERT_BATCH_SIZE = 100

# ---------- Retrieval smoke test ----------
TOP_K = 5

# ---------- Safety switches ----------
CREATE_INDEX_IF_MISSING = True
DRY_RUN = False

## Load API keys

This notebook expects:

- `OPENAI_API_KEY`
- `PINECONE_API_KEY`

You can store them in a `.env` file or set them as environment variables.

In [8]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("PINECONE_API_KEY set:", bool(PINECONE_API_KEY))

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY. Add it to your environment or .env file.")

if not PINECONE_API_KEY:
    raise ValueError("Missing PINECONE_API_KEY. Add it to your environment or .env file.")

OPENAI_API_KEY set: True
PINECONE_API_KEY set: True


## Discover all processed chunk files

In [9]:
chunk_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, CHUNK_GLOB)))
print(f"Found {len(chunk_files)} processed chunk files in {PROCESSED_DIR}")

if not chunk_files:
    raise FileNotFoundError(
        f"No files matched {os.path.join(PROCESSED_DIR, CHUNK_GLOB)}. "
        "Check your processed directory and filename pattern."
    )

chunk_files[:5]

Found 25 processed chunk files in ../data/processed


['../data/processed\\chunks_timestamped_04K0bLwCDdM.json',
 '../data/processed\\chunks_timestamped_1YTKedLQOa0.json',
 '../data/processed\\chunks_timestamped_21G7LA2DcGQ.json',
 '../data/processed\\chunks_timestamped_78K0pbvHzjM.json',
 '../data/processed\\chunks_timestamped_AkX6JqlWRqc.json']

## Load and flatten all chunks

Each file is expected to contain a list of chunk dictionaries like the timestamp-aware format you created earlier, including fields such as:

- `chunk_id`
- `video_id`
- `text`
- `start_time`, `end_time`
- `source_segments`
- `metadata`

In [10]:
all_chunks: List[Dict[str, Any]] = []
file_summaries: List[Dict[str, Any]] = []

iterator = tqdm(chunk_files, desc="Loading files") if tqdm else chunk_files

for path in iterator:
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    if not isinstance(chunks, list):
        raise ValueError(f"Expected a list of chunks in {path}")

    video_id = chunks[0].get("video_id") if chunks else None
    title = chunks[0].get("video_title") if chunks else None

    file_summaries.append({
        "source_file": path,
        "video_id": video_id,
        "video_title": title,
        "chunk_count": len(chunks),
    })

    all_chunks.extend(chunks)

print(f"Loaded {len(all_chunks)} total chunks from {len(chunk_files)} files.")

Loading files:   0%|          | 0/25 [00:00<?, ?it/s]

Loaded 295 total chunks from 25 files.


In [11]:
if pd is not None:
    file_summary_df = pd.DataFrame(file_summaries)
    display(file_summary_df.head())
else:
    print(file_summaries[:3])

,source_file,video_id,video_title,chunk_count
0,../data/processed\chunks_timestamped_04K0bLwCD...,04K0bLwCDdM,The Incredible Properties of Composite Materials,20
1,../data/processed\chunks_timestamped_1YTKedLQO...,1YTKedLQOa0,Understanding Torsion,9
2,../data/processed\chunks_timestamped_21G7LA2Dc...,21G7LA2DcGQ,Understanding Buckling,12
3,../data/processed\chunks_timestamped_78K0pbvHz...,78K0pbvHzjM,Understanding Plane Stress,4
4,../data/processed\chunks_timestamped_AkX6JqlWR...,AkX6JqlWRqc,Understanding True Stress and True Strain,6


## Normalize chunk metadata for Pinecone

This step creates a stable schema for vector storage.

### Design choices for later agent/chat compatibility

Each vector gets:

- a stable **id**
- the raw **text** for embedding
- searchable **metadata**
- timestamp metadata for answer citations
- video/channel info for source attribution
- fields that make future **tool use** easier (filters, source docs, namespaces)

### Important metadata note

Pinecone metadata should stay compact and JSON-friendly.  
This notebook keeps the metadata rich but restrained enough for vector DB usage.

In [12]:
def sanitize_str(value):
    if value is None:
        return None
    value = str(value).strip()
    return value if value else None

def safe_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None

def safe_int(value):
    try:
        if value is None:
            return None
        return int(value)
    except Exception:
        return None

def compact_text(text, max_len=40000):
    text = (text or "").strip()
    return text[:max_len]

def short_hash(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:12]

records = []

for chunk in all_chunks:
    metadata = chunk.get("metadata", {}) or {}
    text = compact_text(chunk.get("text", ""))

    if not text:
        continue

    video_id = sanitize_str(chunk.get("video_id"))
    chunk_id = sanitize_str(chunk.get("chunk_id"))
    title = sanitize_str(chunk.get("video_title"))
    channel = sanitize_str(chunk.get("channel"))
    video_url = sanitize_str(chunk.get("video_url"))

    start_time = safe_float(chunk.get("start_time"))
    end_time = safe_float(chunk.get("end_time"))
    duration_seconds = safe_float(chunk.get("duration_seconds"))

    source_segments = chunk.get("source_segments") or []
    source_segment_count = safe_int(chunk.get("source_segment_count"))

    tags = metadata.get("tags") or []
    categories = metadata.get("categories") or []

    pinecone_id = chunk_id or f"{video_id}_{short_hash(text)}"

    records.append({
        "id": pinecone_id,
        "text": text,
        "metadata": {
            "doc_type": "youtube_transcript_chunk",
            "video_id": video_id,
            "chunk_id": chunk_id,
            "chunk_index": safe_int(chunk.get("chunk_index")),
            "total_chunks": safe_int(chunk.get("total_chunks")),
            "video_title": title,
            "channel": channel,
            "video_url": video_url,
            "start_time": start_time,
            "end_time": end_time,
            "start_time_str": sanitize_str(chunk.get("start_time_str")),
            "end_time_str": sanitize_str(chunk.get("end_time_str")),
            "duration_seconds": duration_seconds,
            "char_count": safe_int(chunk.get("char_count")),
            "word_count": safe_int(chunk.get("word_count")),
            "source_segment_count": source_segment_count,
            "transcript_language": sanitize_str(metadata.get("transcript_language")),
            "transcript_status": sanitize_str(metadata.get("transcript_status")),
            "video_duration": safe_int(metadata.get("video_duration")),
            "upload_date": sanitize_str(metadata.get("upload_date")),
            "view_count": safe_int(metadata.get("view_count")),
            "like_count": safe_int(metadata.get("like_count")),
            "comment_count": safe_int(metadata.get("comment_count")),
            "playlist_index": safe_int(metadata.get("playlist_index")),
            "categories": [str(x) for x in categories if x is not None],
            "tags": [str(x) for x in tags if x is not None],
            "description_preview": sanitize_str((metadata.get("description") or "")[:500]),
            "chunk_text": text,
        },
        "local_source_segments": source_segments,
    })

print(f"Prepared {len(records)} valid records for embedding / upsert.")

Prepared 295 valid records for embedding / upsert.


In [13]:
if pd is not None:
    records_preview = pd.DataFrame([
        {
            "id": r["id"],
            "video_id": r["metadata"]["video_id"],
            "video_title": r["metadata"]["video_title"],
            "chunk_index": r["metadata"]["chunk_index"],
            "char_count": r["metadata"]["char_count"],
            "start_time_str": r["metadata"]["start_time_str"],
            "end_time_str": r["metadata"]["end_time_str"],
        }
        for r in records[:20]
    ])
    display(records_preview.head())
else:
    print(records[:2])

,id,video_id,video_title,chunk_index,char_count,start_time_str,end_time_str
0,04K0bLwCDdM_chunk_0000,04K0bLwCDdM,The Incredible Properties of Composite Materials,0,1178,00:00,01:24
1,04K0bLwCDdM_chunk_0001,04K0bLwCDdM,The Incredible Properties of Composite Materials,1,1189,01:12,02:35
2,04K0bLwCDdM_chunk_0002,04K0bLwCDdM,The Incredible Properties of Composite Materials,2,1139,02:23,04:01
3,04K0bLwCDdM_chunk_0003,04K0bLwCDdM,The Incredible Properties of Composite Materials,3,1194,03:49,05:05
4,04K0bLwCDdM_chunk_0004,04K0bLwCDdM,The Incredible Properties of Composite Materials,4,1191,04:53,06:25


## Corpus sanity checks before embedding

In [14]:
record_ids = [r["id"] for r in records]
duplicate_ids = len(record_ids) - len(set(record_ids))

video_ids = sorted({r["metadata"]["video_id"] for r in records if r["metadata"]["video_id"]})
channels = sorted({r["metadata"]["channel"] for r in records if r["metadata"]["channel"]})

print("Total records:", len(records))
print("Unique record ids:", len(set(record_ids)))
print("Duplicate ids:", duplicate_ids)
print("Unique videos:", len(video_ids))
print("Unique channels:", len(channels))

if duplicate_ids > 0:
    raise ValueError("Duplicate record IDs detected. Fix IDs before upserting to Pinecone.")

Total records: 295
Unique record ids: 295
Duplicate ids: 0
Unique videos: 25
Unique channels: 1


## Create embedding client and Pinecone client

In [15]:
openai_client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Clients created successfully.")

Clients created successfully.


## Create or connect to the Pinecone index

This notebook uses a **bring-your-own-vectors** pattern:

- embeddings are generated with OpenAI
- vectors are stored in a standard Pinecone dense index

That keeps the setup flexible for later notebooks and app code.

In [16]:
existing_index_names = []
index_list = pc.list_indexes()

try:
    existing_index_names = [idx["name"] for idx in index_list]
except Exception:
    try:
        existing_index_names = [idx.name for idx in index_list.indexes]
    except Exception:
        try:
            existing_index_names = [idx.name for idx in index_list]
        except Exception:
            existing_index_names = []

print("Existing indexes:", existing_index_names)

if INDEX_NAME not in existing_index_names:
    if not CREATE_INDEX_IF_MISSING:
        raise ValueError(f"Index '{INDEX_NAME}' does not exist and CREATE_INDEX_IF_MISSING is False.")

    print(f"Creating index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
    )
else:
    print(f"Using existing index: {INDEX_NAME}")

Existing indexes: ['efficient-engineer', 'youtube-rag-mechanical-engineering']
Using existing index: youtube-rag-mechanical-engineering


In [18]:
index = pc.Index(INDEX_NAME)
print("Connected to index:", INDEX_NAME)

Connected to index: youtube-rag-mechanical-engineering


In [19]:
def clean_metadata(metadata):
    cleaned = {}

    for k, v in metadata.items():
        if v is None:
            continue

        if isinstance(v, list):
            cleaned[k] = [str(x) for x in v if x is not None]

        elif isinstance(v, (str, int, float, bool)):
            cleaned[k] = v

        else:
            cleaned[k] = str(v)

    return cleaned

## Embed records in batches

OpenAI embeddings support batched inputs.  
This notebook embeds chunk texts in batches and keeps the metadata aligned.

In [20]:
def batched(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

texts = [r["text"] for r in records]
all_embeddings = []

embed_batches = list(batched(texts, OPENAI_EMBED_BATCH_SIZE))
embed_iterator = tqdm(embed_batches, desc="Embedding batches") if tqdm else embed_batches

for batch_texts in embed_iterator:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=batch_texts,
    )
    batch_embeddings = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embeddings)

print("Embeddings created:", len(all_embeddings))

if len(all_embeddings) != len(records):
    raise ValueError("Mismatch between number of records and number of embeddings.")

Embedding batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings created: 295


In [21]:
vector_dimension = len(all_embeddings[0]) if all_embeddings else None
print("Observed embedding dimension:", vector_dimension)

if vector_dimension != EMBEDDING_DIMENSIONS:
    raise ValueError(
        f"Embedding dimension mismatch: expected {EMBEDDING_DIMENSIONS}, got {vector_dimension}. "
        "Adjust EMBEDDING_DIMENSIONS or the embedding model."
    )

Observed embedding dimension: 1536


## Build Pinecone vectors

In [22]:
vectors = []

for record, embedding in zip(records, all_embeddings):
    cleaned_metadata = clean_metadata(record["metadata"])

    vectors.append({
        "id": record["id"],
        "values": embedding,
        "metadata": cleaned_metadata,
    })

print(f"Prepared {len(vectors)} Pinecone vectors.")
print("Example vector metadata keys:", list(vectors[0]["metadata"].keys())[:12] if vectors else [])

Prepared 295 Pinecone vectors.
Example vector metadata keys: ['doc_type', 'video_id', 'chunk_id', 'chunk_index', 'total_chunks', 'video_title', 'channel', 'video_url', 'start_time', 'end_time', 'start_time_str', 'end_time_str']


In [23]:
null_found = False

for v in vectors:
    for k, val in v["metadata"].items():
        if val is None:
            print("Found null metadata:", k)
            null_found = True

if not null_found:
    print("No null metadata found.")

No null metadata found.


## Upsert vectors into Pinecone

In [24]:
if DRY_RUN:
    print("DRY_RUN=True, so no vectors will be upserted.")
else:
    total_upserted = 0
    upsert_batches = list(batched(vectors, UPSERT_BATCH_SIZE))
    upsert_iterator = tqdm(upsert_batches, desc="Upserting to Pinecone") if tqdm else upsert_batches

    for batch in upsert_iterator:
        index.upsert(vectors=batch, namespace=NAMESPACE)
        total_upserted += len(batch)

    print(f"Upserted {total_upserted} vectors into index='{INDEX_NAME}', namespace='{NAMESPACE}'.")

Upserting to Pinecone:   0%|          | 0/3 [00:00<?, ?it/s]

Upserted 295 vectors into index='youtube-rag-mechanical-engineering', namespace='efficient-engineer-v1'.


## Verify index stats

In [25]:
stats = index.describe_index_stats()
stats

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'efficient-engineer-v1': {'vector_count': 295}},
 'total_vector_count': 295,
 'vector_type': 'dense'}

## Optional smoke-test query

This confirms that:

- the namespace contains vectors
- embeddings can be generated for a user query
- Pinecone returns sensible matches
- timestamps and metadata come back cleanly

You can change the query to something domain-specific from your videos.

In [35]:
TEST_QUERY = "Explain fatigue testing "

query_embedding = openai_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=[TEST_QUERY],
).data[0].embedding

query_response = index.query(
    namespace=NAMESPACE,
    vector=query_embedding,
    top_k=TOP_K,
    include_metadata=True,
)

query_response

{'matches': [{'id': 'o-6V_JoRX1g_chunk_0003',
              'metadata': {'categories': ['Science & Technology'],
                           'channel': 'The Efficient Engineer',
                           'char_count': 1163.0,
                           'chunk_id': 'o-6V_JoRX1g_chunk_0003',
                           'chunk_index': 3.0,
                           'chunk_text': 'large amount of variability in the '
                                         'data. This is typical for fatigue '
                                         'tests even when identical test '
                                         'pieces are used. If we use a best '
                                         'fit S-N curve, as we have done here, '
                                         'there is a significant possibility '
                                         'that our component will fail at a '
                                         'much smaller number of cycles than '
                                   

In [30]:
matches = query_response.get("matches", []) if isinstance(query_response, dict) else getattr(query_response, "matches", [])

formatted_matches = []
for m in matches:
    md_ = m.get("metadata", {}) if isinstance(m, dict) else getattr(m, "metadata", {})
    score = m.get("score") if isinstance(m, dict) else getattr(m, "score", None)

    formatted_matches.append({
        "score": score,
        "video_title": md_.get("video_title"),
        "channel": md_.get("channel"),
        "start_time_str": md_.get("start_time_str"),
        "end_time_str": md_.get("end_time_str"),
        "video_url": md_.get("video_url"),
        "chunk_id": md_.get("chunk_id"),
        "transcript_language": md_.get("transcript_language"),
        "chunk_text": md_.get("chunk_text"),
    })

if pd is not None:
    display(pd.DataFrame(formatted_matches))
else:
    print(formatted_matches)

,score,video_title,channel,start_time_str,end_time_str,video_url,chunk_id,transcript_language,chunk_text
0,0.565327,Understanding Steels and Heat Treatment,The Efficient Engineer,14:37,15:52,https://www.youtube.com/watch?v=VRBpqM6ESrg,VRBpqM6ESrg_chunk_0013,en,or improved material properties. Four key heat...
1,0.499468,Understanding Steels and Heat Treatment,The Efficient Engineer,13:33,14:53,https://www.youtube.com/watch?v=VRBpqM6ESrg,VRBpqM6ESrg_chunk_0012,en,of the main reasons alloying elements like Nic...
2,0.480407,Understanding Steels and Heat Treatment,The Efficient Engineer,15:41,16:57,https://www.youtube.com/watch?v=VRBpqM6ESrg,VRBpqM6ESrg_chunk_0014,en,the furnace. The higher temperature helps bett...
3,0.478137,Understanding Steels and Heat Treatment,The Efficient Engineer,00:00,01:20,https://www.youtube.com/watch?v=VRBpqM6ESrg,VRBpqM6ESrg_chunk_0000,en,If there's one material that defines modern en...
4,0.469309,Understanding Steels and Heat Treatment,The Efficient Engineer,21:10,22:27,https://www.youtube.com/watch?v=VRBpqM6ESrg,VRBpqM6ESrg_chunk_0019,en,followed by quenching to create a hard martens...


In [31]:
# optional direct print view
matches = query_response.get("matches", []) if isinstance(query_response, dict) else getattr(query_response, "matches", [])

for m in matches:
    md_ = m.get("metadata", {}) if isinstance(m, dict) else getattr(m, "metadata", {})
    score = m.get("score") if isinstance(m, dict) else getattr(m, "score", None)

    print("score:", score)
    print("title:", md_.get("video_title"))
    print("time:", md_.get("start_time_str"), "-", md_.get("end_time_str"))
    print("chunk_id:", md_.get("chunk_id"))
    print("chunk_text:", md_.get("chunk_text"))
    print("-" * 100)

score: 0.565326691
title: Understanding Steels and Heat Treatment
time: 14:37 - 15:52
chunk_id: VRBpqM6ESrg_chunk_0013
chunk_text: or improved material properties. Four key heat treatment processes are annealing, normalising, quenching and tempering. Let's see how they work. Annealing is used to make steel soft, ductile, and easy to work with. It's often applied to steel parts after forging or casting to relieve stresses and make the part each to machine. The process involves heating the steel into the austenite region, then cooling it very slowly inside the furnace. Because the cooling is so gradual, the carbon atoms have plenty of time to re-arrange into the wide ferrite–and-cementite layers of coarse pearlite, which has low strength, but is very soft and ductile. Annealing is usually a preparation step before machining or other processing. Normalising is similar to annealing - both heat the steel into the austenite region, with normalising using a slightly higher temperature. The ma

## Save an ingestion manifest

This manifest helps later stages:

- retrieval notebooks
- web app backend
- agent / tool setup
- reproducibility and debugging

In [ ]:
manifest = {
    "generated_at": datetime.now().isoformat(),
    "processed_dir": PROCESSED_DIR,
    "chunk_glob": CHUNK_GLOB,
    "index_name": INDEX_NAME,
    "namespace": NAMESPACE,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimensions": EMBEDDING_DIMENSIONS,
    "record_count": len(records),
    "unique_video_count": len(video_ids),
    "unique_channel_count": len(channels),
    "source_files": file_summaries,
    "sample_record_ids": [r["id"] for r in records[:10]],
# }

# manifest_path = os.path.join(PROCESSED_DIR, MANIFEST_FILENAME)
# with open(manifest_path, "w", encoding="utf-8") as f:
#     json.dump(manifest, f, indent=2, ensure_ascii=False)

# print("Saved manifest to:", manifest_path)

## Recommended next steps

After this notebook succeeds, the next layer should be a **retrieval + chat notebook/app**, not another ingestion notebook.

Recommended progression:

1. retrieval smoke-test notebook  
2. conversational RAG notebook  
3. agentic app with tools + memory  
4. web app deployment  
5. optional observability / evaluation

### Good tool design for your later agent
A useful agentic setup for this dataset usually includes:

- a **retriever tool** over Pinecone
- a **source formatter tool** for timestamps / citations
- conversation **memory**
- optionally a **web / glossary / calculator** tool depending on your domain

### Metadata filters you can use later
Because this notebook stores rich metadata, your app can later filter by:

- `video_id`
- `channel`
- `transcript_language`
- `upload_date`
- `categories`
- `tags`

That will be useful when you build the chat + agent layer.